In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
import torch
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt


# Part 1: Prepare Data for PyTorch (Single Cell)


# 1) Convert numpy arrays to PyTorch tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor  = torch.tensor(y_test, dtype=torch.float32)

print("Converted data to PyTorch tensors")



In [ ]:
# 2. Create TensorDataset objects


train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset  = TensorDataset(X_test_tensor, y_test_tensor)

print("TensorDatasets created")




In [ ]:
# 3. Create DataLoaders


train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print("DataLoaders created")
print("-" * 40)


In [ ]:
# 4. Print shape of one batch


images_batch, ages_batch = next(iter(train_loader))
print("One batch image shape:", images_batch.shape)
print("One batch label shape:", ages_batch.shape)
print("-" * 40)

In [ ]:
# 5. Display sample images

plt.figure()
for i in range(5):
    img = images_batch[i].permute(1, 2, 0)
    plt.subplot(1, 5, i + 1)
    plt.imshow(img)
    plt.title(f"Age: {int(ages_batch[i])}")
    plt.axis("off")

plt.show()


In [ ]:
# Task 1: Write your model class here:

import torch
import torch.nn as nn
import torch.optim as optim


class AgeRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(3 * 36 * 36, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 64)
        self.fc4 = nn.Linear(64, 1)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        x = self.relu(self.fc3(x))
        x = self.fc4(x)
        return x.squeeze(1)





In [ ]:
# Task 2: Write your training loop here:

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for images, ages in loader:
        images = images.to(device)
        ages = ages.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, ages)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# Task 3: Write your validation loop here:

def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0

    with torch.no_grad():
        for images, ages in loader:
            images = images.to(device)
            ages = ages.to(device)

            outputs = model(images)
            loss = criterion(outputs, ages)
            total_loss += loss.item()

    return total_loss / len(loader)

In [ ]:
# Task 4: Define device, model, loss, optimizer:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeRegressionModel().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("Using device:", device)
print("Model, loss, and optimizer initialized")
print("-" * 40)

In [ ]:
# Task 5: Start training for 20 epochs:


num_epochs = 20
train_losses = []
val_losses = []

for epoch in range(1, num_epochs + 1):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss = validate(model, test_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch:02d} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

In [ ]:
# Task 1: Write your code here:
import matplotlib.pyplot as plt
import torch



# Note the blue one is Training Loss and orange is Validation Loss
plt.figure()
plt.plot(train_losses, label="Training Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss (MSE)")
plt.title("Training vs Validation Loss")
plt.show()








In [ ]:

model.eval()

# take one batch from test loader
images, actual_ages = next(iter(test_loader))
images = images.to(device)
actual_ages = actual_ages.to(device)

with torch.no_grad():
    predicted_ages = model(images)

# move to cpu for plotting
images = images.cpu()
actual_ages = actual_ages.cpu()
predicted_ages = predicted_ages.cpu()

plt.figure(figsize=(10, 4))
for i in range(5):
    img = images[i].permute(1, 2, 0)
    plt.subplot(1, 5, i + 1)
    plt.imshow(img)
    plt.title(f"Pred: {predicted_ages[i]:.1f}\nActual: {actual_ages[i]:.0f}")
    plt.axis("off")

plt.suptitle("Predicted Age vs Actual Age")
plt.show()